In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from glob import glob
import rasterio as rio
from rasterio.mask import mask
from rasterio.io import MemoryFile
from PIL import Image
import json
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from src.mslandcover.config import MSTM_PROJ4, LEGEND_CLASSES
from src.mslandcover.utils import raise_if_not_exists
import cv2

In [2]:
# load the shapefilse with boundaries of the regions
shapefiles = glob(r'Z:\guser\dh\NAIP_MS_2023\*\*.shp')
gdfs = []
for shapefile in shapefiles:
    gdf = gpd.read_file(shapefile).to_crs(MSTM_PROJ4) # convert to the same projection
    raster_path = glob(os.path.join(os.path.dirname(shapefile), '*_1m.tif'))[0]
    gdf['raster_path'] = raster_path
    gdfs.append(gdf)

raster_boundaries_gdf = gpd.GeoDataFrame(pd.concat(gdfs))[['raster_path', 'geometry']] # only keep the relevant columns
raster_boundaries_gdf = raster_boundaries_gdf.dissolve(by='raster_path').reset_index() # dissolve the geometries to get the boundaries of the raster
raster_boundaries_gdf.to_file('data/sampling/regions_boundaries.gpkg', driver='GPKG')

In [3]:
# load the samples parquet
raster_boundaries_gdf = gpd.read_file('data/sampling/regions_boundaries.gpkg')
samples = gpd.read_parquet('./data/sampling/samples.par')

# spatial join with the raster boundaries
raster_boundaries_gdf['raster_geometry'] = raster_boundaries_gdf['geometry'] # make a copy of the geometry
samples = gpd.sjoin(samples, raster_boundaries_gdf, predicate='intersects', how='left')

In [5]:
def extract_mask(sample, raster_dataset):
    
    try:
        out_image, out_transform = mask(raster_dataset, [sample['geometry']], crop=True, all_touched=True)
    except Exception as e:
        print(f'type(e) raised while extracting mask for sample {sample.name} in split {sample['split']}: {e}')
        return
    out_meta = raster_dataset.meta.copy()

    if out_image.shape[1] > 256 or out_image.shape[2] > 256:
        # crop the image to 256x256
        out_image = out_image[:, :256, :256]

    filename = str(sample.name)
    if out_image.shape != (3, 256, 256):\
        return
    
    nd_values = np.array([raster_dataset.nodata] * 3)
    pixels_with_nd = np.equal(out_image.transpose(1, 2, 0).reshape(-1, 3), nd_values).all(axis=1)
    if pixels_with_nd.any():
        
        if pixels_with_nd.sum() > 0.05 * len(pixels_with_nd):
            return
    
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform,
    })
    
    out_path = os.path.join('data', 'splits', sample['split'], 'input', filename + '.tif')
    with rio.open(out_path, 'w', **out_meta) as dst:
        dst.write(out_image)
    
    if sample['split'] == 'pretrain' or sample['split'] == 'pretrain_val': # save HSV image for pretraining
        hsv_image = cv2.cvtColor(out_image.transpose(1, 2, 0), cv2.COLOR_RGB2HSV).transpose(2, 0, 1)
        
        target_path = os.path.join('data', 'splits', sample['split'], 'target', filename + '.tif')
        with rio.open(target_path, 'w', **out_meta) as dst:
            dst.write(hsv_image)

def extract_raster(samples_group):
    
    raster_path = samples_group[0]
    with rio.open(raster_path) as raster_dataset:
        samples_group[1].apply(lambda x: extract_mask(x, raster_dataset), axis=1)

n_threads = 8

# sample train, test and val splits first
sub_samples = samples[samples['split'].isin(['train', 'test', 'val'])]
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(extract_raster, list(sub_samples.groupby('raster_path'))))


In [26]:
# convert samples in train, test, val splits to png for annotation
for split in ['train', 'test', 'val']:
    
    os.makedirs(os.path.join('data', 'roboflow', split, 'input'), exist_ok=True)
    os.makedirs(os.path.join('data', 'roboflow', split, 'target'), exist_ok=True)
    samples_split = samples[samples['split'] == split]
    
    sample_files = glob(f'data/splits/{split}/input/*.tif')
    sample_ids = [int(os.path.basename(file).replace('.tif', '')) for file in sample_files]
    
    for id, file in zip(sample_ids, sample_files):
        with rio.open(file) as src:
            img = Image.fromarray(src.read().transpose(1, 2, 0))
            meta = src.meta
        
        # add lat, long of the centroid of the sample to the metadata
        sample = samples_split.loc[id]
        if type(sample) == gpd.GeoDataFrame:
            sample = sample.iloc[0]
        centroid = sample['geometry'].centroid
        meta['lat'] = centroid.y
        meta['lon'] = centroid.x
        
        # from histogram vector, add the relative frequency of each class to the metadata
        hist = sample['hist_vector']
        legend_classes = LEGEND_CLASSES.copy()
        legend_classes.pop(0) # remove nodata class
        
        meta['class_freq'] = {legend_classes[i+1]: hist[i] for i in range(len(hist))}

        # need to convert CRS to string such that it is serializable
        meta['crs'] = meta['crs'].to_string()
        
        with open(f'data/png_images/{split}/input/{id}.json', 'w') as f:
            json.dump(meta, f, indent=4)
        
        img.save(f'data/png_images/{split}/input/{id}.png')

In [ ]:
# sample pretrain and pretrain_val splits
sub_samples = samples[samples['split'].isin(['pretrain', 'pretrain_val'])]
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(extract_raster, list(sub_samples.groupby('raster_path')))
)

<!-- ## Sampling Points for Accuracy Assesment -->